In [1]:
# import libraries
from openai import OpenAI
import time
from dotenv import load_dotenv
import os
import sys
from pathlib import Path
import torch
import json
import random
import re
import sys
from sklearn.metrics import classification_report as sklearn_classification_report
from seqeval.metrics import classification_report as seqeval_classification_report

# set path to project root and import custom classes and functions
base_path = Path.cwd() / "../../../"
sys.path.append(str(base_path.resolve()))
from utils.evaluation import extract_spans, mention_level_evaluation

In [2]:
def create_llm_annotations(text, spans):

    # sort the spans according to their start and end index
    spans = sorted(spans, key=lambda x: x["start"])

    # store the text and set index variable
    llm_text = ""
    last_idx = 0

    # loop through spans and add the span with custom characters
    for span in spans:
        llm_text += text[last_idx:span["start"]]
        llm_text += f"@@{text[span["start"]:span["end"]]}##"
        last_idx = span["end"]

    # add the rest of the text
    llm_text += text[last_idx:]

    return llm_text

def llm_output_to_bio(annotated_text):

    # split words via a regex
    words = re.findall(r"@@.*?##|\w+|'\w+|[^\w\s]", annotated_text)

    # empty list to store the bio tags
    bio_tags = []

    # loop through all words
    for word in words:

        # if it is an annotated span, split words and assign bio labels
        if word.startswith("@@") and word.endswith("##"):
            entity_text = word[2:-2]
            entity_words = re.findall(r"\w+|'\w+|[^\w\s]", entity_text)
            for i, t in enumerate(entity_words):
                tag = "B-sg" if i == 0 else "I-sg"
                bio_tags.append((t, tag))
        else:
            # otherwise assign O tag
            bio_tags.append((word, "O"))

    return bio_tags

In [3]:
# initialize empty dataset list
dataset = []

with open("../../../01_data/annotations/annotations_no_augmentations.json", "r") as f:
    data = json.load(f)

# loop through all sentences in the data
for task in data:
    # get the sentence and all annotations
    text = task["sentence"]
    spans = task["annotations"]
    labels = [annotation["text"] for annotation in spans]
   
    llm_text = create_llm_annotations(text, spans)
    bio_tags = llm_output_to_bio(llm_text)

    # add everything to the dataset list
    dataset.append({
        "text": text,
        "labels": labels,
        "llm_text": llm_text,
        "bio_tags": bio_tags
    })

In [4]:
# load the model
load_dotenv()  # reads .env file
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [5]:
def compile_ner_prompt(few_shot_examples, test_sentence):
    chat = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant that extracts *social group mentions* from text.\n\n"
                "### Definition of a Social Group\n"
                "A social group is a segment of society or a collection of people who share common socio-demographic traits "
                "or attributes that are either ascriptive (e.g., gender, ethnicity) or acquired (e.g., education, occupation). "
                "Relevant traits include sex, gender, age, ethnicity, language, religion, nationality, place of residence, "
                "income, occupation, and education.\n\n"
                "### Exclusions\n"
                "- Do **not** include implicit social group references such as *people*, *everyone*, *communities*, *the public*, or *the nation*.\n"
                "- Exclude **institutional or organizational entities** (e.g., trade unions, political parties, police departments, companies).\n"
                "- However, you **may include groupings within those institutions** if the defining feature is socio-demographic "
                "(e.g., *workers*, *union members*, *police officers*, *teachers*, *business owners*).\n"
                "- Exclude groupings defined primarily by shared beliefs, ideology, or political affiliation.\n\n"
                "### Task\n"
                "Identify and mark all social group mentions in the provided sentence.\n\n"
                "### Output Format\n"
                "- Return the full sentence.\n"
                "- Mark the **start** of each social group mention with `@@` and the **end** with `##`.\n"
                "- If there are no social group mentions, just respond with the full sentence without changing anything."
            )
        }
    ]

    # add few-shot examples
    for example in few_shot_examples:
        chat.append({"role": "user", "content": f"Sentence: {example['text']}"})
        chat.append({"role": "assistant", "content": example["llm_text"]})
    
    # add the test sentence
    chat.append({"role": "user", "content": f"Sentence: {test_sentence}"})

    return chat

In [6]:
# create some few-shot examples
non_empty_examples = [ex for ex in dataset if ex["labels"]]
empty_examples = [ex for ex in dataset if not ex["labels"]]
few_shot_examples = random.sample(non_empty_examples, 4) + random.sample(empty_examples, 1)

# create test dataset
split_idx = int(len(non_empty_examples)*0.97)
test_dataset = non_empty_examples[split_idx:] + random.sample(empty_examples, int(len(empty_examples)*0.01))
random.shuffle(test_dataset)

In [7]:
test_sentence = "I congratulate the hon. friend and his family."
prompt = compile_ner_prompt(few_shot_examples, test_sentence)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=prompt
)

output_text = response.choices[0].message.content.strip()
output_text

'I congratulate the hon. friend and his @@family##.'

In [24]:
# generate the answers for the normal format and store in a list
gen_answers = []

for i in range(len(test_dataset)):
    sentence = test_dataset[i]["text"]
    prompt = compile_ner_prompt(few_shot_examples, sentence)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=prompt
        )
    output_text = response.choices[0].message.content.strip()
    
    # convert the generated prediction to the bio scheme
    output_bio = llm_output_to_bio(output_text)
    #answer_list = to_list_or_empty(answer)
    gen_answers.append({"text": output_text,
                        "bio": output_bio})

In [29]:
# show a few predictions
for idx in range(10, 20):
    input = test_dataset[idx]["text"]
    prediction = gen_answers[idx]["text"]
    print(f"Input: {input}")
    print(f"prediction: {prediction}")
    print("-"*100)

Input: The temporary system does require them, and that causes delays and adds to congestion.
prediction: The temporary system does require them, and that causes delays and adds to congestion.
----------------------------------------------------------------------------------------------------
Input: I am sure that both sides of the House genuinely appreciate the excellent work done by all staff in our NHS, which at a time of unprecedented strain relies more than ever on the goodwill of its employees to keep going.
prediction: I am sure that both sides of the House genuinely appreciate the excellent work done by all @@staff## in our @@NHS##, which at a time of unprecedented strain relies more than ever on the goodwill of its @@employees## to keep going.
----------------------------------------------------------------------------------------------------
Input: What recent discussions he has had with his counterpart in China on the persecution of Christians in that country.
prediction: Wh

In [27]:
# evaluate the generated answers

# get list of bio tags only
ground_truth_bio = [[tag for (_, tag) in sent["bio_tags"]] for sent in test_dataset]
pred_bio = [[tag for (_, tag) in sent["bio"]] for sent in gen_answers]

filtered_gt = []
filtered_pred = []
for gt, pred in zip(ground_truth_bio, pred_bio):
    if len(gt) == len(pred):
        filtered_gt.append(gt)
        filtered_pred.append(pred)

y_true = [tag for sent in filtered_gt for tag in sent]
y_pred = [tag for sent in filtered_pred for tag in sent]

# evaluate at the word level
print(sklearn_classification_report(y_true, y_pred))

# evaluate at the entity level with seqeval
print(seqeval_classification_report(filtered_gt, filtered_pred))

              precision    recall  f1-score   support

        B-sg       0.58      0.89      0.70       101
        I-sg       0.48      0.35      0.41       117
           O       0.97      0.96      0.97      2675

    accuracy                           0.94      2893
   macro avg       0.68      0.74      0.69      2893
weighted avg       0.94      0.94      0.94      2893

              precision    recall  f1-score   support

          sg       0.43      0.66      0.52       101

   micro avg       0.43      0.66      0.52       101
   macro avg       0.43      0.66      0.52       101
weighted avg       0.43      0.66      0.52       101



In [28]:
all_true_spans = []
all_predicted_spans = []

for idx in range(len(filtered_gt)):

    # get the spans
    all_true_spans.append(extract_spans(filtered_gt[idx]))
    all_predicted_spans.append(extract_spans(filtered_pred[idx]))

# apply cross-span evaluation
mention_level_evaluation(all_true_spans, all_predicted_spans)

{'precision': 0.5213371003996004,
 'recall': 0.5736458333333333,
 'f1': 0.5177331349206349}